In [1]:
# ── Сводная таблица baseline (test) ───────────────────────────────────────────
# Читает results/baseline/{model}_test.json по всем 4 моделям, строит:
#   1) таблицу агрегатных метрик,
#   2) таблицу per-class recall (контраст линейная vs нелинейные),
#   3) сохраняет объединённый summary.json.

import sys; sys.path.insert(0, "..")
import json
from src import config

MODELS = [
    ("logreg", "LogReg"),
    ("rf",     "RandomForest"),
    ("xgb",    "XGBoost"),
    ("mlp",    "MLP"),
]

# Загружаем метрики.
results = {}
for key, name in MODELS:
    path = config.BASELINE_DIR / f"{key}_test.json"
    if not path.exists():
        print(f"[warn] нет файла: {path}")
        continue
    with open(path) as f:
        results[name] = json.load(f)

# ── 1. Агрегатные метрики ─────────────────────────────────────────────────────
print("=" * 78)
print("  BASELINE — TEST (порог заморожен на val)")
print("=" * 78)
print(f"\n  {'Модель':<14}{'MCC':>9}{'F1_macro':>10}{'PR-AUC':>9}"
      f"{'ROC-AUC':>9}{'FPR@95':>9}{'порог':>8}")
print("  " + "-" * 66)
for _, name in MODELS:
    if name not in results:
        continue
    m = results[name]
    print(f"  {name:<14}{m['MCC']:>9.4f}{m['F1_macro']:>10.4f}{m['PR_AUC']:>9.4f}"
          f"{m['ROC_AUC']:>9.4f}{m['FPR_at_TPR95']:>9.4f}{m['threshold']:>8.4f}")

# ── 2. Per-class recall ───────────────────────────────────────────────────────
# Порядок классов — по убыванию размера (из любой модели, n одинаков).
any_model = next(iter(results.values()))
classes = sorted(any_model["per_class_recall"].items(),
                 key=lambda x: -x[1]["n"])

print("\n" + "=" * 78)
print("  PER-CLASS RECALL (test)")
print("=" * 78)
header = f"\n  {'класс':<26}{'n':>9}" + "".join(f"{name:>13}" for _, name in MODELS)
print(header)
print("  " + "-" * (35 + 13 * len(MODELS)))
for cls, d in classes:
    row = f"  {cls:<26}{d['n']:>9,}"
    for key, name in MODELS:
        if name not in results:
            row += f"{'—':>13}"
            continue
        r = results[name]["per_class_recall"].get(cls, {}).get("recall")
        row += f"{r:>12.2%} " if r is not None else f"{'—':>13}"
    print(row)

# ── 3. Сохраняем объединённый summary ─────────────────────────────────────────
summary = {
    "dataset": "LycoS-Unicas-IDS2018",
    "split":   "stratified random 70/15/15, shuffled",
    "n_test":  any_model["n_rows"],
    "threshold_policy": "tuned on val, frozen on test",
    "models": {name: results[name] for _, name in MODELS if name in results},
}
out = config.BASELINE_DIR / "summary.json"
with open(out, "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(f"\n→ summary сохранён: {out}")

  BASELINE — TEST (порог заморожен на val)

  Модель              MCC  F1_macro   PR-AUC  ROC-AUC   FPR@95   порог
  ------------------------------------------------------------------
  LogReg           0.9383    0.9684   0.9720   0.9930   0.0327  0.6175
  RandomForest     0.9970    0.9985   0.9986   0.9997   0.0005  0.3900
  XGBoost          0.9970    0.9985   0.9994   0.9998   0.0004  0.7613
  MLP              0.9961    0.9981   0.9967   0.9996   0.0006  0.9175

  PER-CLASS RECALL (test)

  класс                             n       LogReg RandomForest      XGBoost          MLP
  ---------------------------------------------------------------------------------------
  Benign                    1,500,000      96.67%       99.84%       99.84%       99.82% 
  DoS Hulk                    270,445     100.00%      100.00%      100.00%      100.00% 
  DDoS HOIC                   161,157     100.00%      100.00%      100.00%       99.99% 
  DDoS LOIC-HTTP               43,400     100.00%     

In [ ]:
import duckdb
con = duckdb.connect()
f = "../data/raw/LycoS-IDS-2017/Wednesday-WorkingHours.pcap_lycos.csv"
dist = con.execute(f"""
    SELECT label, COUNT(*) AS n
    FROM read_csv_auto('{f}')
    GROUP BY label ORDER BY n DESC
""").df()
print(dist.to_string(index=False))